# Pandas — Tipos de datos y conversión

Guía para **inspeccionar** dtypes y **convertir** columnas de un tipo a otro.

| Notebook relacionado | Contenido |
|----------------------|-----------|
| [04.01](04.01-pandas-dataframes-series.ipynb) | `dtypes`, `info()`, verificación con `isinstance` |
| [04.03](04.03-pandas-data-manipulation.ipynb) | `fillna`, marcadores en CSV, ceros como faltantes; `replace` / `map` |
| [07.07](../07-scikit-learn/07.07-transformacion-datos-por-tipo.ipynb) | Qué hacer con cada tipo antes de sklearn |


In [ ]:
import pandas as pd
import numpy as np

## 1. Inspeccionar tipos

- **dtype de una columna:** `df['col'].dtype`
- **Todas las columnas:** `df.dtypes` o `df.info()`
- **Filtrar por tipo:** `df.select_dtypes(include=[...])` / `exclude=[...]`
- **Comprobar familia:** `pd.api.types.is_numeric_dtype(s)`, `is_string_dtype`, `is_datetime64_any_dtype`, `is_categorical_dtype`


In [ ]:
df = pd.DataFrame({
    "id": [1, 2, 3],
    "precio": [10.5, 20.0, 15.75],
    "ciudad": ["Madrid", "Barcelona", "Valencia"],
    "activo": [True, False, True],
    "fecha": ["2024-01-15", "2024-02-01", "2024-03-10"],
})

df.dtypes
df.info()

df.select_dtypes(include=[np.number])      # int64, float64...
df.select_dtypes(include=["object", "string"])
df.select_dtypes(exclude=[np.number])

pd.api.types.is_numeric_dtype(df["precio"])
pd.api.types.is_string_dtype(df["ciudad"])

## 2. Valores nulos según el tipo de dato

En pandas conviven **varios marcadores de ausencia**. No todos se detectan igual ni conviven con todos los dtypes.

### Marcadores habituales

| Marcador | Qué es | `isna()` / `isnull()` | Dónde aparece |
|----------|--------|------------------------|---------------|
| **`None`** | Ausencia en Python | ✅ Sí | Columnas `object`, listas al crear el DataFrame |
| **`np.nan`** | “Not a Number” (float IEEE) | ✅ Sí | `float64`, columnas numéricas tras `to_numeric(..., errors='coerce')` |
| **`pd.NA`** | Ausencia genérica (extension types) | ✅ Sí | `Int64`, `string`, `boolean`, `Float64`… |
| **`NaT`** | “Not a Time” | ✅ Sí | `datetime64`, `timedelta64` tras `to_datetime` / `to_timedelta` |

> `isna()` y `isnull()` son **equivalentes**. `notna()` / `notnull()` son la negación.

### Qué nulo “encaja” con cada dtype

| dtype | ¿Admite faltantes nativamente? | Marcador típico | Comentario |
|-------|--------------------------------|-----------------|------------|
| **`int64`** | ❌ No | — | Si metes `None`/`NaN`, pandas **promociona** a `float64` con `NaN` |
| **`float64`** | ✅ Sí | `np.nan` | Único estándar en numéricos clásicos |
| **`object`** | ✅ Sí | `None`, a veces `np.nan` mezclado | Texto + `None` es muy habitual en CSV mal tipados |
| **`string`** (nullable) | ✅ Sí | `pd.NA` | Preferible a `object` para texto |
| **`Int64`**, **`Float64`**, **`boolean`** | ✅ Sí | `pd.NA` | Enteros/bool **con** huecos sin pasar a float |
| **`datetime64`** | ✅ Sí | `NaT` | `pd.NaT` es el alias habitual |
| **`timedelta64`** | ✅ Sí | `NaT` | Igual que fechas |
| **`category`** | ✅ Sí | `NaN` / `pd.NA` | La categoría faltante no cuenta como nivel de la categoría |
| **`bool`** (clásico) | ⚠️ Limitado | — | Solo `True`/`False`; para tercer estado usa `boolean` nullable |

### Lo que **no** es nulo para pandas (hasta que lo conviertas)

Cadenas como `""`, `"N/A"`, `"NA"`, `"null"`, `"unknown"`, espacios… **no** activan `isna()` solas. Hay que normalizarlas a `np.nan` / `pd.NA` con `replace`, `na_values` al leer CSV o `to_*` con `errors='coerce'`. Ver [04.03](04.03-pandas-data-manipulation.ipynb).

### Al convertir tipos: qué pasa con los nulos

| Conversión | Efecto sobre faltantes |
|------------|------------------------|
| `None` → columna numérica | Suele acabar como `np.nan` (`float64`) |
| `astype('int64')` con NaN | **Error** — usar `Int64` o `fillna` antes |
| `astype('Int64')` | `np.nan` / `None` → `pd.NA` |
| `to_numeric(..., errors='coerce')` | Inválidos y muchos textos → `np.nan` |
| `to_datetime(..., errors='coerce')` | Inválidos → `NaT` |
| `map` con dict incompleto | Valores no mapeados → **`NaN`** (a diferencia de `replace`) |


In [ ]:
# Detección unificada
marcadores = [None, np.nan, pd.NA, pd.NaT]
pd.Series(marcadores).isna()   # todos True

# None en object vs NaN en float
s_obj = pd.Series([1, None, 3], dtype=object)
s_flo = pd.Series([1.0, np.nan, 3.0])
s_obj.isna()   # True en el None
s_flo.isna()   # True en el NaN

# int64 no admite NA: pandas sube a float64
pd.Series([1, None, 3]).dtype   # float64

# Int64 nullable: pd.NA
pd.Series([1, pd.NA, 3], dtype="Int64")

# Fechas: NaT
pd.to_datetime(["2024-01-01", "no-fecha"], errors="coerce")  # NaT en la segunda

# Texto "N/A" NO es nulo hasta convertir
txt = pd.Series(["10", "N/A", "20"])
txt.isna().sum()                              # 0
pd.to_numeric(txt, errors="coerce").isna()    # 1 (la fila "N/A")

# Conteo de nulos por columna
df_n = pd.DataFrame({"a": [1, None, 3], "b": ["x", "", "y"]})
df_n.isna().sum()
df_n.replace("", np.nan).isna().sum()         # "" tratado como faltante

## 3. Tabla resumen: de un tipo a otro

| Origen (ejemplo) | Objetivo | Método habitual |
|------------------|----------|-----------------|
| Texto numérico (`"12.5"`, `"25 años"`) | `float` / `int` | `pd.to_numeric(..., errors='coerce')` |
| Texto fecha (`"2024-01-15"`) | `datetime64` | `pd.to_datetime(..., errors='coerce')` |
| Texto duración | `timedelta64` | `pd.to_timedelta(..., errors='coerce')` |
| `float64` con decimales | `int` | `astype('int64')` o `round()` antes si hace falta |
| `int64` | `float` | `astype('float64')` |
| Cualquier columna | `str` | `astype(str)` o `astype('string')` |
| 0/1 o `True`/`False` | `bool` | `astype('bool')` |
| Strings repetidos (ciudad, talla) | `category` | `astype('category')` o `pd.Categorical(...)` |
| Con NaN en enteros | entero nullable | `astype('Int64')` (mayúscula) |
| Con NaN en texto | string nullable | `astype('string')` |
| Varios tipos a la vez | inferir extension dtypes | `df.convert_dtypes()` |

> **`astype`**: conversión directa cuando los valores ya son compatibles.  
> **`to_numeric` / `to_datetime`**: parsing + errores → `NaN`/`NaT` con `errors='coerce'`.


## 4. `astype` — conversión directa

En **Serie** o **columna**. Devuelve copia salvo que reasignes: `df['col'] = df['col'].astype(...)`.


In [ ]:
s = pd.Series([1, 2, 3, 4])

s.astype('float64')
s.astype(int)              # alias de int64 en muchos casos
s.astype(str)              # '1', '2', '3'
s.astype('bool')           # cualquier no-cero → True

# Varias columnas a la vez
df2 = df.copy()
df2 = df2.astype({"id": "int64", "precio": "float32"})
df2.dtypes

### Tipos nullable (con `pd.NA`)

Los dtypes clásicos (`int64`, `float64`) **no** admiten enteros faltantes; usan `float` con NaN. Los **extension types** sí:

| dtype | Uso |
|-------|-----|
| `Int64`, `Float64`, `boolean` | Numéricos / bool con `pd.NA` |
| `string` | Texto con `pd.NA` (preferible a `object` en datos nuevos) |


In [ ]:
s_na = pd.Series([1, pd.NA, 3], dtype="Int64")
s_na.dtype

df3 = pd.DataFrame({"a": [1, 2, np.nan]})
df3["a_int"] = df3["a"].astype("Int64")   # NaN → pd.NA
df3.dtypes

## 5. `pd.to_numeric`, `to_datetime`, `to_timedelta`

Para **parsear** texto o valores mixtos. Parámetro clave: `errors`.

| `errors` | Comportamiento |
|----------|----------------|
| `'raise'` | Falla si hay valor inválido (por defecto) |
| `'coerce'` | Inválidos → `NaN` / `NaT` |
| `'ignore'` | Devuelve la entrada sin convertir si falla |


In [ ]:
edad_txt = pd.Series(["25", "30", "N/A", "  40  "]])
pd.to_numeric(edad_txt, errors="coerce")     # 'N/A' → NaN

precio_txt = pd.Series(["12,5", "10"])       # coma decimal europea
pd.to_numeric(precio_txt.str.replace(",", ".", regex=False), errors="coerce")

fechas = pd.Series(["2024-01-15", "15/03/2024", "no-es-fecha"])
pd.to_datetime(fechas, errors="coerce", dayfirst=True)  # NaT en la inválida

duracion = pd.Series(["2 days", "3 hours", "bad"])
pd.to_timedelta(duracion, errors="coerce")

## 6. Categóricas (`category`)

Ahorra memoria y deja claro que hay **pocas etiquetas repetidas**. Para orden lógico (S &lt; M &lt; L), define categorías y orden.


In [ ]:
tallas = pd.Series(["S", "M", "L", "M", "S"])
cat = tallas.astype("category")
cat.dtype
cat.cat.categories

# Orden explícito (ordinal)
orden = ["S", "M", "L"]
cat_ord = pd.Categorical(tallas, categories=orden, ordered=True)
pd.Series(cat_ord).cat.codes   # 0, 1, 2... según orden

df["talla_cat"] = df["ciudad"].astype("category")  # ejemplo en DataFrame

## 7. `convert_dtypes()` — inferencia automática

Pandas intenta pasar columnas a dtypes **extension** (`Int64`, `string`, `boolean`…) detectando NaN y texto.


In [ ]:
df_raw = pd.DataFrame({
    "n": [1, 2, None],
    "t": ["a", "b", "c"],
    "b": [True, False, True],
})
df_raw.dtypes

df_inf = df_raw.convert_dtypes()
df_inf.dtypes   # p. ej. Int64, string, boolean

## 8. Booleanos y texto → número (mapeo)

Si no es un cast de tipo sino **sustituir etiquetas** (`"sí"` → `1`), usa `replace` / `map` en [04.03](04.03-pandas-data-manipulation.ipynb). Después aplica `astype` si quieres un dtype concreto.


In [ ]:
resp = pd.Series(["si", "no", "si"])
MAPEO = {"si": 1, "no": 0}
resp_num = resp.replace(MAPEO).astype("int64")
resp_num

## 9. Errores frecuentes

| Error / síntoma | Causa | Qué hacer |
|-----------------|-------|-----------|
| `Cannot convert non-finite values` al castear a `int` | Hay `NaN` en la columna | `astype('Int64')` o `fillna` antes de `int64` |
| Columna sigue `object` tras leer CSV | Números guardados como texto | `pd.to_numeric(..., errors='coerce')` |
| Fechas como `object` | Formato no detectado | `pd.to_datetime` con `format=` o `dayfirst=True` |
| `SettingWithCopyWarning` | Cadena sobre vista | `.copy()` y luego convertir |
| sklearn no acepta `object` / strings | Faltan dummies o numéricos | Codificar o `to_numeric`; ver [07.07](../07-scikit-learn/07.07-transformacion-datos-por-tipo.ipynb) |

## Checklist rápido

1. `df.dtypes` / `df.info()` y `df.isna().sum()` (None / NaN / NA / NaT)
2. Elegir `astype` vs `to_*` según la tabla
3. `errors='coerce'` al parsear datos sucios
4. Nullable (`Int64`, `string`) si necesitas `pd.NA`
5. `category` para pocas etiquetas repetidas
6. Antes de ML: columnas numéricas o codificadas; sin `object` crudo en features
